# Baseline v5 — Medium-Hard Negative Mining + Retrain Cross-Encoder

## Fix so với v5 cũ

| | v5 cũ (thất bại) | **v5 fix (Nhóm A)** |
|--|--|--|
| Negatives từ | rank 1-20 (quá hard) | **rank 15-30 (medium-hard)** |
| Vấn đề | CE confused vì neg quá giống pos | CE học phân biệt vừa tầm |
| `SKIP_TOP_K` | 0 | **14** |
| `TOP_MINE` | 20 | **30** |

**Nguyên tắc:** Rank 1-14 của v4 bi-encoder đã quá gần với positive (ngữ nghĩa gần như giống nhau). CE không thể học gì từ boundary đó. Dùng rank 15-30 — đủ khó để CE học, nhưng vẫn khác biệt đủ để phân biệt.

## Cell 0 — Config & Imports

In [ ]:
import json, csv, time, random, gc
import numpy as np
import faiss
import torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

# ── Paths ──
ROOT          = Path(".")
DATA_DIR      = ROOT / "data"
EVAL_DIR      = ROOT / "outputs" / "eval"
TMP_DIR       = ROOT / "outputs" / "tmp"
MDL_DIR       = ROOT / "outputs" / "models"

TRAIN_FILE    = DATA_DIR / "train.jsonl"
DEV_FILE      = DATA_DIR / "dev.jsonl"
TRAIN_NEG     = DATA_DIR / "train_with_neg.jsonl"
EVAL_QA_FILE  = EVAL_DIR / "eval_qa.jsonl"

FT_BI_PATH    = MDL_DIR  / "legal_hf_finetuned" / "final"
FAISS_V4      = TMP_DIR  / "faiss_v4.index"
MAP_V4        = TMP_DIR  / "faiss_mapping_v4.jsonl"

HN_TRAIN_FILE = EVAL_DIR / "hard_neg_train_v5fix.jsonl"
CE_V5_DIR     = MDL_DIR  / "cross_encoder_v5fix"
RERANK_CSV_V4 = EVAL_DIR / "rerank_metrics_v4.csv"
RERANK_CSV_V5 = EVAL_DIR / "rerank_metrics_v5.csv"    # overwrite v5 cũ

# ── CE retraining config ──
BASE_CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_EPOCHS     = 5
CE_BATCH      = 32
CE_MAX_LEN    = 256
SEED          = 42

# ── Mining config (KEY FIX) ──
TOP_MINE      = 30    # retrieve top-30 (cần nhiều hơn để có rank 15-30)
SKIP_TOP_K    = 14    # bỏ top-14 (quá hard / quá giống positive)
HARD_NEG_PER  = 2     # lấy tối đa 2 medium-hard negatives/query

TOP_N_EVAL    = 50
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
CE_V5_DIR.mkdir(parents=True, exist_ok=True)

print(f"torch  : {torch.__version__}")
print(f"Device : {DEVICE}")
print(f"Mining : top-{TOP_MINE}, skip top-{SKIP_TOP_K}, take {HARD_NEG_PER}/query")
print(f"  → sử dụng rank {SKIP_TOP_K+1} → {TOP_MINE} làm medium-hard negatives")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path, max_rows=None):
    rows, errors = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows: break
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: errors += 1
    if errors: print(f"  ⚠ {errors} malformed lines in {Path(path).name}")
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_hit(faiss_id, expected_citations, mapping):
    row = mapping[faiss_id]
    for ec in expected_citations:
        ci = ec.get("chunk_index", -2)
        if ci != -1 and row["chunk_index"] == ci: return True
        if (row["van_ban"] == ec.get("van_ban", "") and
            row["dieu"]    == ec.get("dieu",    "") and
            row["khoan"]   == ec.get("khoan",   "")): return True
    return False

def is_positive_meta(cand, meta):
    if cand["van_ban"] == meta.get("van_ban","") and cand["van_ban"] != "":
        if (cand["dieu"]  == meta.get("dieu","") and
            cand["khoan"] == meta.get("khoan","")): return True
    ci = meta.get("chunk_index", -2)
    if ci != -1 and cand["chunk_index"] == ci: return True
    return False

def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

print("Utilities loaded ✓")

## Cell 2 — Load v4 Bi-Encoder + FAISS

In [ ]:
print(f"Loading v4 fine-tuned bi-encoder: {FT_BI_PATH}")
ft_bi     = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4  = faiss.read_index(str(FAISS_V4))
mapping_v4= load_jsonl(MAP_V4)
print(f"Bi-encoder ✓ | dim={ft_bi.get_sentence_embedding_dimension()}")
print(f"FAISS ✓ | {index_v4.ntotal} vectors, {len(mapping_v4)} mapping entries")

## Cell 3 — Medium-Hard Negative Mining

```
Top-30 kết quả của v4 bi-encoder:

rank  1-14:  BỎ QUA  ← quá gần positive (v5 cũ dùng vùng này → thất bại)
rank 15-30:  LẤY LÀM NEGATIVE ← medium-hard (v5 fix)
```

In [ ]:
train_rows = load_jsonl(TRAIN_NEG)
pos_rows   = [r for r in train_rows if r.get("label") == 1]
random.seed(SEED); random.shuffle(pos_rows)
print(f"Positive rows: {len(pos_rows)}")

hn_train = []
stats    = {"pos": 0, "medium_hard": 0, "no_neg": 0}

for r in tqdm(pos_rows, desc="Mining medium-hard negatives"):
    query = r.get("query", "").strip()
    pos_p = r.get("passage", "").strip()
    meta  = r.get("meta", {})
    if not query or not pos_p: continue

    # Luôn thêm positive
    hn_train.append({"query": query, "passage": pos_p, "label": 1, "type": "positive"})
    stats["pos"] += 1

    # Retrieve top-30
    q_emb  = ft_bi.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_MINE)
    ids    = ids[0].tolist()

    # ── KEY FIX: Bỏ top SKIP_TOP_K, lấy từ rank SKIP_TOP_K+1 ──
    medium_hard_pool = ids[SKIP_TOP_K:]   # rank 15-30

    added = 0
    for fid in medium_hard_pool:
        if fid < 0 or added >= HARD_NEG_PER: break
        cand = mapping_v4[fid]
        # Bỏ qua nếu vẫn là positive
        if cand["passage"] == pos_p: continue
        if is_positive_meta(cand, meta): continue

        hn_train.append({"query": query, "passage": cand["passage"],
                         "label": 0, "type": "medium_hard_neg",
                         "neg_rank": ids.index(fid) + 1})
        added += 1
        stats["medium_hard"] += 1

    if added == 0: stats["no_neg"] += 1

write_jsonl(HN_TRAIN_FILE, hn_train)
print(f"\n── Mining complete ──")
print(f"  Positive pairs    : {stats['pos']}")
print(f"  Medium-hard negs  : {stats['medium_hard']}  ({stats['medium_hard']/max(stats['pos'],1):.1f}/query avg)")
print(f"  No neg found      : {stats['no_neg']} queries")
print(f"  Total CE train    : {len(hn_train)} rows")
print(f"  Pos ratio         : {stats['pos']/len(hn_train):.1%}")
print(f"  Saved → {HN_TRAIN_FILE}")

## Cell 4 — Build Dev set với medium-hard negatives

In [ ]:
dev_rows = load_jsonl(DEV_FILE)
random.seed(SEED); random.shuffle(dev_rows)

hn_dev = []
for r in tqdm(dev_rows[:500], desc="Dev medium-hard negatives"):
    query = r.get("query", "").strip()
    pos_p = r.get("passage", "").strip()
    meta  = r.get("meta", {})
    if not query or not pos_p: continue

    hn_dev.append({"query": query, "passage": pos_p, "label": 1})

    q_emb  = ft_bi.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_MINE)
    ids    = ids[0].tolist()

    for fid in ids[SKIP_TOP_K:]:       # rank 15-30
        if fid < 0: break
        cand = mapping_v4[fid]
        if cand["passage"] == pos_p: continue
        if is_positive_meta(cand, meta): continue
        hn_dev.append({"query": query, "passage": cand["passage"], "label": 0})
        break

random.seed(SEED); random.shuffle(hn_dev)
n_pos = sum(1 for r in hn_dev if r["label"]==1)
n_neg = sum(1 for r in hn_dev if r["label"]==0)
print(f"Dev set: {len(hn_dev)} rows | pos={n_pos}, neg={n_neg}")

## Cell 5 — Retrain Cross-Encoder với Medium-Hard Negatives
> ⏱️ ~15-30 phút (RTX 3050 Ti, 5 epochs)

In [ ]:
# Giải phóng bi-encoder trước khi train CE
del ft_bi; gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None
print("VRAM cleared ✓")

# Chuẩn bị samples
random.seed(SEED); random.shuffle(hn_train)
train_samples = [InputExample(texts=[r["query"], r["passage"]], label=float(r["label"]))
                 for r in hn_train if "query" in r and "passage" in r]
dev_samples   = [InputExample(texts=[r["query"], r["passage"]], label=float(r["label"]))
                 for r in hn_dev]

print(f"Train: {len(train_samples)} | Dev: {len(dev_samples)}")
print(f"Pos ratio: {sum(1 for s in train_samples if s.label==1)/len(train_samples):.1%}")

# Load base CE (bắt đầu từ pretrained, không dùng v1)
use_fp16 = (DEVICE == "cuda")
ce_v5    = CrossEncoder(BASE_CE_MODEL, num_labels=1, max_length=CE_MAX_LEN, device=DEVICE)
evaluator= CEBinaryClassificationEvaluator.from_input_examples(dev_samples, name="dev_v5fix")
warmup   = int(len(train_samples) / CE_BATCH * CE_EPOCHS * 0.1)

print(f"\nTraining CE v5fix: {CE_EPOCHS} epochs, batch={CE_BATCH}, warmup={warmup}...")
t0 = time.time()

ce_v5.fit(
    train_dataloader=DataLoader(train_samples, shuffle=True, batch_size=CE_BATCH),
    evaluator=evaluator,
    epochs=CE_EPOCHS,
    warmup_steps=warmup,
    output_path=str(CE_V5_DIR),
    use_amp=use_fp16,
)

elapsed = round((time.time()-t0)/60, 1)
print(f"Training done in {elapsed} min")

saved_path = CE_V5_DIR / "saved_model"
ce_v5.save(str(saved_path))

cfg = {"base_model": BASE_CE_MODEL, "epochs": CE_EPOCHS, "batch": CE_BATCH,
       "skip_top_k": SKIP_TOP_K, "top_mine": TOP_MINE, "hard_neg_per": HARD_NEG_PER,
       "train_samples": len(train_samples), "minutes": elapsed}
(CE_V5_DIR / "config.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"CE v5fix saved → {saved_path}")

## Cell 6 — Evaluate v5fix: v4_FT_Bi + v5fix_CE

In [ ]:
# Reload bi-encoder
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None

print(f"Reloading v4 bi-encoder: {FT_BI_PATH}")
ft_bi     = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4  = faiss.read_index(str(FAISS_V4))
mapping_v4= load_jsonl(MAP_V4)
eval_qa   = load_jsonl(EVAL_QA_FILE)
print(f"Loaded ✓ | Eval: {len(eval_qa)} questions")

r_base   = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}
r_rerank = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}

for item in tqdm(eval_qa, desc="Evaluate v5fix"):
    query = item["query"]; ec = item["expected_citations"]
    q_emb = ft_bi.encode([query], normalize_embeddings=True,
                          convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_N_EVAL)
    ids    = ids[0].tolist()

    # Baseline (v4)
    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_base[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in ids[:k] if i>=0) else 0)
    mrr = 0.0
    for rank,i in enumerate(ids[:10],1):
        if i>=0 and is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_base["MRR@10"].append(mrr)

    # Rerank với CE v5fix
    cands   = [(mapping_v4[i]["passage"],i) for i in ids if i>=0]
    rscores = ce_v5.predict([[query,c[0]] for c in cands], batch_size=32) if cands else []
    ranked  = sorted(zip(rscores,[c[1] for c in cands]),reverse=True)
    r_ids   = [r[1] for r in ranked]

    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_rerank[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in r_ids[:k]) else 0)
    mrr = 0.0
    for rank,i in enumerate(r_ids[:10],1):
        if is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_rerank["MRR@10"].append(mrr)

print("\n── v5fix Results ──")
print(f"  {'Metric':<10} {'v4 Baseline':>14} {'v5fix+Rerank':>14} {'Δ':>8}")
print("  " + "-"*50)
for k, key in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]:
    b  = avg(r_base[k]); re = avg(r_rerank[k])
    sign = "+" if re-b>=0 else ""
    print(f"  {key:<10} {b:>14.4f} {re:>14.4f} {sign}{re-b:>7.4f}")

## Cell 7 — So sánh v4 vs v5fix & Lưu CSV

In [ ]:
v4 = {}
if RERANK_CSV_V4.exists():
    with open(RERANK_CSV_V4, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            v4[row["metric"]] = {
                "base":   float(row.get("v4_finetuned", 0)),
                "rerank": float(row.get("v4_reranked",  0)),
            }

v5b = {"Recall@1":avg(r_base["R@1"]),   "Recall@3":avg(r_base["R@3"]),
       "Recall@5":avg(r_base["R@5"]),   "MRR@10":avg(r_base["MRR@10"])}
v5r = {"Recall@1":avg(r_rerank["R@1"]), "Recall@3":avg(r_rerank["R@3"]),
       "Recall@5":avg(r_rerank["R@5"]), "MRR@10":avg(r_rerank["MRR@10"])}

print("\n" + "="*104)
print(f"  {'Metric':<10} {'v4(FT_Bi)':>13} {'v4+OldCE':>12} {'v5(FT_Bi)':>12} {'v5+MH_CE':>12} {'Δ(v5R-v4R)':>13}")
print("="*104)
for metric in ["Recall@1","Recall@3","Recall@5","MRR@10"]:
    v4b_  = v4.get(metric,{}).get("base",  float("nan"))
    v4r_  = v4.get(metric,{}).get("rerank",float("nan"))
    v5b_  = v5b[metric]; v5r_ = v5r[metric]
    delta = v5r_ - v4r_
    sign  = "+" if delta >= 0 else ""
    print(f"  {metric:<10} {v4b_:>13.4f} {v4r_:>12.4f} {v5b_:>12.4f} {v5r_:>12.4f} {sign}{delta:>12.4f}")
print("="*104)

rows_v5 = [
    {"metric":"Recall@1","v4_finetuned":v4.get("Recall@1",{}).get("base",""),"v4_reranked":v4.get("Recall@1",{}).get("rerank",""),"v5_finetuned":v5b["Recall@1"],"v5_mh_reranked":v5r["Recall@1"]},
    {"metric":"Recall@3","v4_finetuned":v4.get("Recall@3",{}).get("base",""),"v4_reranked":v4.get("Recall@3",{}).get("rerank",""),"v5_finetuned":v5b["Recall@3"],"v5_mh_reranked":v5r["Recall@3"]},
    {"metric":"Recall@5","v4_finetuned":v4.get("Recall@5",{}).get("base",""),"v4_reranked":v4.get("Recall@5",{}).get("rerank",""),"v5_finetuned":v5b["Recall@5"],"v5_mh_reranked":v5r["Recall@5"]},
    {"metric":"MRR@10",  "v4_finetuned":v4.get("MRR@10",{}).get("base",  ""),"v4_reranked":v4.get("MRR@10",{}).get("rerank",  ""),"v5_finetuned":v5b["MRR@10"],  "v5_mh_reranked":v5r["MRR@10"]},
]
with open(RERANK_CSV_V5, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["metric","v4_finetuned","v4_reranked","v5_finetuned","v5_mh_reranked"])
    w.writeheader(); w.writerows(rows_v5)

print(f"\nSaved → {RERANK_CSV_V5} ✓")